# Notebook 6: Urgency Scoring & Prioritization

**Goal:** Score each complaint 0-100 and assign to priority buckets (P1/P2/P3/P4)

**Input:** `data/processed/grievance_processed.csv` — must contain `sentiment_label` + `model_confidence` written by Notebook 5

**Outputs:**
- `outputs/grievance_with_urgency.csv` — all complaints scored and sorted by urgency
- `outputs/urgency_config.json` — weights, thresholds, keywords, statistics

Combines 4 factors with specific weights:

| Factor | Weight | Source |
|--------|--------|--------|
| Sentiment label | 40% | NB05 model prediction |
| Emergency keywords | 25% | Rule-based safety net |
| Model confidence | 20% | NB05 model confidence |
| Recency | 15% | Complaint filed date |



##  Import Libraries

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

✅ Libraries imported successfully
Pandas version: 2.2.2
NumPy version: 2.0.2


##  Configuration: Weights, Thresholds & Priority Rules

In [2]:
# Factor weights — must sum to 1.0
WEIGHTS = {
    'sentiment'  : 0.40,   # 40% — AI model sentiment label (most reliable signal)
    'keywords'   : 0.25,   # 25% — Emergency keywords (safety net if model fails)
    'confidence' : 0.20,   # 20% — Model confidence in its prediction
    'recency'    : 0.15    # 15% — How recently complaint was filed
}

total_weight = sum(WEIGHTS.values())
print(f'Total weight: {total_weight}')
assert abs(total_weight - 1.0) < 0.001, f'Weights must sum to 1.0, got {total_weight}'
print('Weights validated')

# Sentiment label → base score (0-100)
SENTIMENT_SCORES = {
    'critical' : 100,
    'negative' : 60,
    'neutral'  : 20,
    'positive' : 0
}

print('\nSentiment Scores:')
for label, score in SENTIMENT_SCORES.items():
    print(f'  {label:12} -> {score:3d} points')

# Priority tiers — score ranges and SLAs
PRIORITY_TIERS = {
    'P1': {'score_min': 80, 'score_max': 100, 'sla': '2 hours'},
    'P2': {'score_min': 60, 'score_max': 79,  'sla': '24 hours'},
    'P3': {'score_min': 40, 'score_max': 59,  'sla': '3 days'},
    'P4': {'score_min': 0,  'score_max': 39,  'sla': '7 days'}
}

print('\nPriority Tiers:')
for tier, info in PRIORITY_TIERS.items():
    print(f'  {tier} ({info["score_min"]:2d}-{info["score_max"]:2d}) -> Respond within {info["sla"]}')

Total weight: 1.0
Weights validated

Sentiment Scores:
  critical     -> 100 points
  negative     ->  60 points
  neutral      ->  20 points
  positive     ->   0 points

Priority Tiers:
  P1 (80-100) -> Respond within 2 hours
  P2 (60-79) -> Respond within 24 hours
  P3 (40-59) -> Respond within 3 days
  P4 ( 0-39) -> Respond within 7 days


##  Load Sentiment Model Output from Notebook 5

In [3]:
BASE_DIR    = Path('../')
DATA_DIR    = BASE_DIR / 'data'
MODELS_DIR  = BASE_DIR / 'models'
OUTPUTS_DIR = BASE_DIR / 'outputs'

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

try:
    df = pd.read_csv(DATA_DIR / 'processed' / 'grievance_processed.csv')
    print(f'Loaded processed data: {df.shape[0]:,} rows x {df.shape[1]} columns')
except FileNotFoundError:
    print('grievance_processed.csv not found, trying cleaned data instead')
    df = pd.read_csv(DATA_DIR / 'cleaned' / 'grievance_cleaned.csv')
    print(f'Loaded cleaned data: {df.shape[0]:,} rows x {df.shape[1]} columns')

print('\nFirst 2 rows:')
print(df.head(2))
print(f'\nColumns ({len(df.columns)}):')
print(df.columns.tolist())

✅ Loaded processed data: 24,774 rows × 21 columns

First 2 rows:
   Unique Key         Created Date          Closed Date  \
0    32274996  2015-12-26 23:03:56  2015-12-27 01:27:37   
1    32252575  2015-12-22 19:43:42  2015-12-22 21:31:10   

                       Agency Name       Complaint Type        Descriptor  \
0  New York City Police Department        Public Notice  Area Information   
1  New York City Police Department  Information Request  Area Information   

     Location Type   Borough  Status  \
0  Street/Sidewalk    QUEENS  Closed   
1  Street/Sidewalk  BROOKLYN  Closed   

                             Resolution Description  ...  resolution_hours  \
0                 No violations or issues detected.  ...          2.394722   
1  The situation is under control and satisfactory.  ...          1.791111   

   hour_created  day_of_week      closed_date_fmt resolution_hours_fmt  \
0            23     Saturday  2015-12-27 01:27:37              2.4 hrs   
1            19      

## Define Emergency Keyword Tiers

In [4]:
# Keyword tier system — safety net if model misclassifies
EMERGENCY_KEYWORDS = {
    'tier_1': {
        'keywords'    : ['fire', 'collapse', 'trapped', 'explosion', 'drown', 'suffocation'],
        'points'      : 40,
        'description' : 'Life-threatening / immediate danger'
    },
    'tier_2': {
        'keywords'    : ['urgent', 'emergency', 'danger', 'live wire', 'electrocute', 'hazard'],
        'points'      : 25,
        'description' : 'High risk / serious concern'
    },
    'tier_3': {
        'keywords'    : ['broken', 'garbage', 'blocked', 'not working', 'flooding', 'leak', 'hazardous'],
        'points'      : 10,
        'description' : 'Infrastructure / service issue'
    }
}

print('Emergency Keyword Tiers:')
for tier, info in EMERGENCY_KEYWORDS.items():
    print(f'\n{tier.upper()} ({info["points"]} pts): {info["description"]}')
    print(f'  Keywords: {info["keywords"]}')

Emergency Keyword Tiers:

TIER_1 (40 pts): Life-threatening / immediate danger
  Keywords: ['fire', 'collapse', 'trapped', 'explosion', 'drown', 'suffocation']

TIER_2 (25 pts): High risk / serious concern
  Keywords: ['urgent', 'emergency', 'danger', 'live wire', 'electrocute', 'hazard']

TIER_3 (10 pts): Infrastructure / service issue
  Keywords: ['broken', 'garbage', 'blocked', 'not working', 'flooding', 'leak', 'hazardous']


##  Build Scoring Functions

In [5]:
def get_sentiment_score(sentiment_label):
    """Convert sentiment label to 0-100 score. Defaults to neutral (20) if missing."""
    if pd.isna(sentiment_label):
        return 20
    return SENTIMENT_SCORES.get(str(sentiment_label).lower().strip(), 20)


def get_keyword_score(text):
    """Check for emergency keywords and return score (0-40)."""
    if pd.isna(text):
        return 0
    text_lower = str(text).lower()

    # Tier 1 — life-threatening: return immediately if matched
    for keyword in EMERGENCY_KEYWORDS['tier_1']['keywords']:
        if keyword in text_lower:
            return EMERGENCY_KEYWORDS['tier_1']['points']

    # Tier 2 — high risk
    score = 0
    for keyword in EMERGENCY_KEYWORDS['tier_2']['keywords']:
        if keyword in text_lower:
            score = max(score, EMERGENCY_KEYWORDS['tier_2']['points'])

    # Tier 3 — infrastructure (only if no tier 2 matched)
    if score == 0:
        for keyword in EMERGENCY_KEYWORDS['tier_3']['keywords']:
            if keyword in text_lower:
                score = max(score, EMERGENCY_KEYWORDS['tier_3']['points'])

    return score


def get_confidence_score(confidence):
    """Convert model confidence to 0-100 score.
    
    Accepts both 0-1 scale (raw softmax) and 0-100 scale (percentage).
   
    Defaults to 50 (medium confidence) if missing.
    """
    if pd.isna(confidence) or confidence is None:
        return 50
    confidence = float(confidence)
    if confidence <= 1.0:           # 0-1 scale — convert to 0-100
        confidence = confidence * 100
    return min(100, max(0, confidence))


def get_recency_score(created_date, reference_date=None):
    """Calculate recency score (0-100). Today = 100, 30+ days ago = 0.
    
     Strips timezone info from both dates before comparison
    to avoid TypeError when CSV dates are tz-aware and now() is tz-naive.
    """
    if pd.isna(created_date):
        return 0

    if reference_date is None:
        reference_date = pd.Timestamp.now()
    else:
        reference_date = pd.Timestamp(reference_date)

    created_date   = pd.Timestamp(created_date)


    if created_date.tzinfo is not None:
        created_date = created_date.tz_localize(None)
    if reference_date.tzinfo is not None:
        reference_date = reference_date.tz_localize(None)

    days_old = (reference_date - created_date).days

    if days_old < 0:    # Future date — treat as today
        return 100

    return max(0, 100 - (days_old / 30 * 100))


def calculate_urgency_score(row, reference_date=None):
    """
    Main scoring function combining all 4 factors.

    Formula:
        urgency = (sentiment x 0.40) + (keywords x 0.25) + (confidence x 0.20) + (recency x 0.15)

    Args:
        row           : pandas Series (or dict) with complaint data
        reference_date: date to calculate recency against (default: today)

    Returns:
        dict with component scores and final urgency_score (0-100)
    """
    # Use combined_text if available, else fall back to clean_text
    text = row.get('combined_text') or row.get('clean_text', '')

    sentiment_score  = get_sentiment_score(row.get('sentiment_label', 'neutral'))
    keyword_score    = get_keyword_score(text)
    confidence_score = get_confidence_score(row.get('model_confidence', None))
    recency_score    = get_recency_score(row.get('Created Date', None), reference_date)

    urgency = (
        (sentiment_score  * WEIGHTS['sentiment'])  +
        (keyword_score    * WEIGHTS['keywords'])   +
        (confidence_score * WEIGHTS['confidence']) +
        (recency_score    * WEIGHTS['recency'])
    )

    return {
        'sentiment_score'  : sentiment_score,
        'keyword_score'    : keyword_score,
        'confidence_score' : confidence_score,
        'recency_score'    : recency_score,
        'urgency_score'    : min(100, max(0, urgency))
    }


def get_priority_tier(urgency_score):
    """Convert urgency score to priority tier (P1/P2/P3/P4)."""
    for tier, info in PRIORITY_TIERS.items():
        if info['score_min'] <= urgency_score <= info['score_max']:
            return tier
    return 'P4'   # Default to lowest priority


print('All scoring functions defined successfully')

All scoring functions defined successfully


## Test on Sample Complaints

In [6]:
test_df = df.head(6).copy()


if 'sentiment_label' not in test_df.columns:
    raise RuntimeError(
        'sentiment_label column missing from dataframe.\n'
        'Run Notebook 5 Cell 13 first to generate real model predictions.'
    )

if 'model_confidence' not in test_df.columns:
    raise RuntimeError(
        'model_confidence column missing from dataframe.\n'
        'Run Notebook 5 Cell 13 first to generate real model predictions.'
    )

print('SAMPLE URGENCY SCORING TEST')
print(f'Using real NB05 model predictions for {len(test_df)} complaints')

for idx, (i, row) in enumerate(test_df.iterrows(), 1):
    scores   = calculate_urgency_score(row)
    priority = get_priority_tier(scores['urgency_score'])

   
    unique_key    = row.get('Unique Key', row.get('unique_key', 'N/A'))
    clean_text    = str(row.get('clean_text', ''))[:60]
    sentiment_lbl = row.get('sentiment_label', 'N/A')

    print(f'\nCOMPLAINT #{idx} (ID: {unique_key})')
    print(f'   Text     : {clean_text}...')
    print(f'   Sentiment: {sentiment_lbl}  |  Confidence: {row.get("model_confidence", "N/A")}')
    print(f'   -----------------------------------------------')
    print(f'   Sentiment Score    : {scores["sentiment_score"]:6.2f}  (40%)')
    print(f'   Keyword Score      : {scores["keyword_score"]:6.2f}  (25%)')
    print(f'   Confidence Score   : {scores["confidence_score"]:6.2f}  (20%)')
    print(f'   Recency Score      : {scores["recency_score"]:6.2f}  (15%)')
    print(f'   ===============================================')
    print(f'   URGENCY SCORE      : {scores["urgency_score"]:6.2f}')
    print(f'   PRIORITY TIER      : {priority}  (SLA: {PRIORITY_TIERS[priority]["sla"]})')

print('\nSample test completed successfully')

SAMPLE URGENCY SCORING TEST
Using real NB05 model predictions for 6 complaints

📋 COMPLAINT #1 (ID: 32274996)
   Text: public notice area violation issue...
   Sentiment: positive
   ─────────────────────────────────
   Sentiment Score    :   0.00 (40% weight)
   Keyword Score      :   0.00 (25% weight)
   Confidence Score   :  85.00 (20% weight)
   Recency Score      :   0.00 (15% weight)
   ═════════════════════════════════
   🎯 URGENCY SCORE   :  17.00
   🚨 PRIORITY TIER   : P4 (SLA: 7 days)

📋 COMPLAINT #2 (ID: 32252575)
   Text: area situation control satisfactory...
   Sentiment: neutral
   ─────────────────────────────────
   Sentiment Score    :  20.00 (40% weight)
   Keyword Score      :   0.00 (25% weight)
   Confidence Score   :  92.00 (20% weight)
   Recency Score      :   0.00 (15% weight)
   ═════════════════════════════════
   🎯 URGENCY SCORE   :  26.40
   🚨 PRIORITY TIER   : P4 (SLA: 7 days)

📋 COMPLAINT #3 (ID: 31768878)
   Text: community update everything appears goo

## Score All Complaints

In [7]:
print(f'Starting to score {len(df):,} complaints...')
print('This may take a minute or two...')

scores_list = []


# Original used idx+1 which is the dataframe index — could be non-sequential
for row_num, (idx, row) in enumerate(df.iterrows(), 1):
    if row_num % 2000 == 0:
        print(f'  Processed {row_num:,} / {len(df):,} complaints...')
    scores_list.append(calculate_urgency_score(row))

scores_df    = pd.DataFrame(scores_list)
df_scored    = pd.concat([df.reset_index(drop=True), scores_df.reset_index(drop=True)], axis=1)
df_scored['priority_tier'] = df_scored['urgency_score'].apply(get_priority_tier)
df_scored_sorted = df_scored.sort_values('urgency_score', ascending=False).reset_index(drop=True)

print(f'\nScoring completed!')
print(f'  Total complaints scored : {len(df_scored_sorted):,}')
print(f'  Total columns           : {len(df_scored_sorted.columns)}')


key_col  = 'Unique Key'   if 'Unique Key'   in df_scored_sorted.columns else df_scored_sorted.columns[0]
display_cols = [key_col, 'urgency_score', 'priority_tier', 'sentiment_score', 'keyword_score', 'confidence_score']
print(f'\nTop 5 rows (highest urgency):')
print(df_scored_sorted[display_cols].head())

Starting to score 24,774 complaints...
This may take a minute or two...



  Processed 2,000 / 24,774 complaints...
  Processed 4,000 / 24,774 complaints...


  Processed 6,000 / 24,774 complaints...
  Processed 8,000 / 24,774 complaints...


  Processed 10,000 / 24,774 complaints...
  Processed 12,000 / 24,774 complaints...


  Processed 14,000 / 24,774 complaints...


  Processed 16,000 / 24,774 complaints...
  Processed 18,000 / 24,774 complaints...


  Processed 20,000 / 24,774 complaints...


  Processed 22,000 / 24,774 complaints...
  Processed 24,000 / 24,774 complaints...



✅ Scoring completed!
   Total complaints scored: 24,774
   Columns: 27

First 5 rows (highest urgency):
   Unique Key  urgency_score priority_tier  sentiment_score  keyword_score  \
0    31525979          24.25            P4               20             25   
1    30367415          24.25            P4               20             25   
2    30132433          24.25            P4               20             25   
3    31569508          24.25            P4               20             25   
4    32238523          24.25            P4               20             25   

   confidence_score  
0             99.97  
1             99.97  
2             99.97  
3             99.97  
4             99.97  


## Save Results

In [8]:
output_csv = OUTPUTS_DIR / 'grievance_with_urgency.csv'
df_scored_sorted.to_csv(output_csv, index=False)

print(f'Saved: {output_csv}')
print(f'  File size : {output_csv.stat().st_size / 1024 / 1024:.2f} MB')
print(f'  Rows      : {len(df_scored_sorted):,}')
print(f'  Columns   : {len(df_scored_sorted.columns)}')

✅ Saved: ../outputs/grievance_with_urgency.csv
   File size: 11.42 MB
   Rows: 24,774
   Columns: 27


## Priority Distribution Report

In [9]:
print('PRIORITY DISTRIBUTION REPORT')

priority_dist    = df_scored_sorted['priority_tier'].value_counts().sort_index()
total_complaints = len(df_scored_sorted)

print('\nComplaints by Priority Tier:')
for tier in ['P1', 'P2', 'P3', 'P4']:
    count = priority_dist.get(tier, 0)
    pct   = (count / total_complaints) * 100
    sla   = PRIORITY_TIERS[tier]['sla']
    bar   = 'x' * int(pct // 3)
    print(f'  {tier} ({PRIORITY_TIERS[tier]["score_min"]:2d}-{PRIORITY_TIERS[tier]["score_max"]:2d}):'
          f'  {count:6,} ({pct:5.2f}%)  ->  {sla:10s}  {bar}')

print(f'  -----------------------------------------------')
print(f'  TOTAL : {total_complaints:,} complaints')

print('\nUrgency Score Statistics:')
print(f'  Min    : {df_scored_sorted["urgency_score"].min():.2f}')
print(f'  Max    : {df_scored_sorted["urgency_score"].max():.2f}')
print(f'  Mean   : {df_scored_sorted["urgency_score"].mean():.2f}')
print(f'  Median : {df_scored_sorted["urgency_score"].median():.2f}')
print(f'  Std    : {df_scored_sorted["urgency_score"].std():.2f}')


key_col       = 'Unique Key'    if 'Unique Key'    in df_scored_sorted.columns else df_scored_sorted.columns[0]
type_col      = 'Complaint Type' if 'Complaint Type' in df_scored_sorted.columns else None

top_cols = [key_col, 'priority_tier', 'urgency_score']
if type_col:
    top_cols.append(type_col)
top_cols.append('clean_text')

top_10 = df_scored_sorted.head(10)[top_cols].copy()
top_10['clean_text'] = top_10['clean_text'].str[:50]

print('\nTop 10 Most Urgent Complaints:')
print(top_10.to_string(index=False))

PRIORITY DISTRIBUTION REPORT

Complaints by Priority Tier:
  P1 (80-100):      0 complaints ( 0.00%) → 2 hours
  P2 (60-79):      0 complaints ( 0.00%) → 24 hours
  P3 (40-59):      0 complaints ( 0.00%) → 3 days
  P4 ( 0-39): 24,774 complaints (100.00%) → 7 days
  ─────────────────────────────────
  TOTAL                : 24,774 complaints

Urgency Score Statistics:
  Min Score  : 18.00
  Max Score  : 24.25
  Mean Score : 19.05
  Median Score : 18.00
  Std Dev    : 1.94

Top 10 Most Urgent Complaints:
 Unique Key priority_tier  urgency_score          Complaint Type                                         clean_text
   31525979            P4          24.25 Noise - Street/Sidewalk noise street sidewalk loud talking forwarded non e
   30367415            P4          24.25         Illegal Parking illegal parking commercial overnight parking forwa
   30132433            P4          24.25         Illegal Parking illegal parking commercial overnight parking forwa
   31569508            P4   

##  Save Configuration for Production

In [10]:
config = {
    'created_at'  : datetime.now().isoformat(),
    'description' : 'Complaints urgency scoring and prioritization configuration',
    'version'     : '1.1',
    'weights'     : WEIGHTS,
    'sentiment_scores' : SENTIMENT_SCORES,
    'priority_tiers'   : {
        tier: {
            'score_min': info['score_min'],
            'score_max': info['score_max'],
            'sla'      : info['sla']
        }
        for tier, info in PRIORITY_TIERS.items()
    },
    'emergency_keywords' : EMERGENCY_KEYWORDS,
    'formula' : 'urgency = (sentiment x 0.40) + (keywords x 0.25) + (confidence x 0.20) + (recency x 0.15)',
    'statistics' : {
        'total_complaints'    : int(len(df_scored_sorted)),
        'urgency_score_min'   : float(df_scored_sorted['urgency_score'].min()),
        'urgency_score_max'   : float(df_scored_sorted['urgency_score'].max()),
        'urgency_score_mean'  : float(df_scored_sorted['urgency_score'].mean()),
        'urgency_score_median': float(df_scored_sorted['urgency_score'].median()),
        'priority_distribution': {
            tier: int(priority_dist.get(tier, 0))
            for tier in ['P1', 'P2', 'P3', 'P4']
        }
    }
}

config_path = OUTPUTS_DIR / 'urgency_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Saved configuration: {config_path}')
print(json.dumps(config, indent=2)[:600] + '...')

✅ Saved configuration: ../outputs/urgency_config.json

Configuration saved:
{
  "created_at": "2026-04-03T09:39:56.702768",
  "description": "Complaints urgency scoring and prioritization configuration",
  "version": "1.1",
  "weights": {
    "sentiment": 0.4,
    "keywords": 0.25,
    "confidence": 0.2,
    "recency": 0.15
  },
  "sentiment_scores": {
    "critical": 100,
    "negative": 60,
    "neutral": 20,
    "positive": 0
  },
  "priority_tiers": {
    "P1": {
      "score_min": 80,
      "score_max": 100,
      "sla": "2 hours"
    },
    "P2": {
      "score_min": 60,
      "score_max": 79,
      "sla": "24 hours"
    },
    "P3": {
      "score_min": 40,
   ...


## Live Scoring Function for New Complaints

Use this to score any new complaint in real time after both notebooks have run.

In [11]:
def score_new_complaint(
    complaint_text  : str,
    sentiment_label : str   = 'neutral',
    model_confidence: float = 50.0,
    created_date            = None
) -> dict:
    """
    Score a single new complaint in real time.

    Args:
        complaint_text  : raw or cleaned complaint text
        sentiment_label : NB05 model output — one of: positive / neutral / negative / critical
        model_confidence: NB05 model confidence.
                          Accepts 0-1 scale (e.g. 0.95) OR 0-100 scale (e.g. 95.0).
                          NB05  writes 0-100 scale.
                          Default: 50.0 (medium confidence)
        created_date    : when the complaint was filed (str or datetime). Default: now.

    Returns:
        dict with urgency_score, priority_tier, sla, and component_scores
    """
    if created_date is None:
        created_date = pd.Timestamp.now()

    fake_row = {
        'combined_text'    : complaint_text,
        'clean_text'       : complaint_text,
        'sentiment_label'  : sentiment_label,
        'model_confidence' : model_confidence,
        'Created Date'     : created_date
    }

    scores   = calculate_urgency_score(fake_row)
    priority = get_priority_tier(scores['urgency_score'])

    return {
        'complaint_text'   : complaint_text,
        'sentiment_label'  : sentiment_label,
        'model_confidence' : model_confidence,
        'created_date'     : str(created_date),
        'urgency_score'    : round(scores['urgency_score'], 2),
        'priority_tier'    : priority,
        'sla'              : PRIORITY_TIERS[priority]['sla'],
        'component_scores' : {
            'sentiment'  : round(scores['sentiment_score'],  2),
            'keywords'   : round(scores['keyword_score'],    2),
            'confidence' : round(scores['confidence_score'], 2),
            'recency'    : round(scores['recency_score'],    2)
        }
    }


# Test on 3 example complaints
print('LIVE SCORING FUNCTION TEST')

test_cases = [
    {'text': 'There is a fire in the building and people are trapped inside',
     'sentiment': 'critical', 'confidence': 95.0},
    {'text': 'The garbage bin has not been collected for a week',
     'sentiment': 'neutral',  'confidence': 70.0},
    {'text': 'There is an emergency with a live wire on the street',
     'sentiment': 'negative', 'confidence': 88.0},
]

for i, test in enumerate(test_cases, 1):
    result = score_new_complaint(test['text'], test['sentiment'], test['confidence'])
    print(f'\nTest Case #{i}:')
    print(f"  Text      : {result['complaint_text'][:65]}")
    print(f"  Sentiment : {result['sentiment_label']}  |  Confidence: {result['model_confidence']}")
    print(f"  Urgency   : {result['urgency_score']}  ->  {result['priority_tier']} (SLA: {result['sla']})")
    print(f"  Breakdown : {result['component_scores']}")

print('\nLive scoring function works correctly')

LIVE SCORING FUNCTION TEST

Test Case #1:
  Text      : There is a fire in the building and people are trapped inside
  Sentiment : critical  |  Confidence: 95.0
  Urgency   : 84.0  ->  P1 (SLA: 2 hours)
  Breakdown : {'sentiment': 100, 'keywords': 40, 'confidence': 95.0, 'recency': 100.0}

Test Case #2:
  Text      : The garbage bin has not been collected for a week
  Sentiment : neutral  |  Confidence: 70.0
  Urgency   : 39.5  ->  P4 (SLA: 7 days)
  Breakdown : {'sentiment': 20, 'keywords': 10, 'confidence': 70.0, 'recency': 100.0}

Test Case #3:
  Text      : There is an emergency with a live wire on the street
  Sentiment : negative  |  Confidence: 88.0
  Urgency   : 62.85  ->  P2 (SLA: 24 hours)
  Breakdown : {'sentiment': 60, 'keywords': 25, 'confidence': 88.0, 'recency': 100.0}

Live scoring function works correctly


## Final Summary

In [12]:
print('=' * 65)
print('  NOTEBOOK 6 COMPLETE — OUTPUT FILES')
print('=' * 65)

print(f'  grievance_with_urgency.csv')
print(f'    Location : {output_csv}')
print(f'    Size     : {output_csv.stat().st_size / 1024 / 1024:.2f} MB')
print(f'    Rows     : {len(df_scored_sorted):,}')
print(f'    Columns  : {len(df_scored_sorted.columns)}')
print()
print(f'  urgency_config.json')
print(f'    Location : {config_path}')
print(f'    Contains : Weights, Thresholds, Keywords, Stats')
print()
print('  Priority summary:')
for tier in ['P1', 'P2', 'P3', 'P4']:
    count = int(priority_dist.get(tier, 0))
    pct   = count / total_complaints * 100
    print(f'    {tier}: {count:,} complaints ({pct:.1f}%) — SLA {PRIORITY_TIERS[tier]["sla"]}')
print('=' * 65)

OUTPUT FILES GENERATED:
  📊 grievance_with_urgency.csv
     Location: ../outputs/grievance_with_urgency.csv
     Size: 11.42 MB
     Rows: 24,774

  ⚙️  urgency_config.json
     Location: ../outputs/urgency_config.json
     Contains: Weights, Thresholds, Keywords, Stats
